In [ ]:
# Enterprise-grade dual CSV chunker

import csv
from pathlib import Path

BASE = Path(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\energy_project_data")

FILES = {
    "demand": BASE / "demand.csv",
    "price": BASE / "price.csv"
}

CHUNK_SIZE = 500_000

def chunk_file(label, input_path):
    outdir = BASE / f"{label}_chunks"
    outdir.mkdir(exist_ok=True)

    with input_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)

        file_index = 1
        row_count = 0
        out = None
        writer = None

        for row in reader:
            if row_count % CHUNK_SIZE == 0:
                if out:
                    out.close()
                out = (outdir / f"{label}.part{file_index}.csv").open(
                    "w", newline="", encoding="utf-8"
                )
                writer = csv.writer(out)
                writer.writerow(header)
                file_index += 1

            writer.writerow(row)
            row_count += 1

        if out:
            out.close()

    print(f"✓ Chunked {label}: {row_count} rows → {file_index-1} files")

def main():
    for label, path in FILES.items():
        chunk_file(label, path)

if __name__ == "__main__":
    main()
